# Installation, Importing and Initialization

In [ ]:
#install all required packages
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

#list of required packages
packages = [
    'groq',
    'supabase',
    'sentence-transformers>=2.2.0',  #for embeddings
    'torch',
    'transformers>=4.34.0'
]

for package in packages:
    install_package(package)
    print(f"{package} installed")

In [ ]:
#import all libraries
import os
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq
from supabase import create_client, Client
import json
import re
from datetime import datetime
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#import kaggle secrets
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

#supabase client setup
#retrieve supabase credentials from kaggle secrets
SUPABASE_URL = user_secrets.get_secret("SUPABASE_URL")
SUPABASE_KEY = user_secrets.get_secret("SUPABASE_KEY")
#initialize supabase client
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Supabase client initialized")


#groq 
#retrieve groq api key from kaggle secrets
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
#initialize groq client
if not GROQ_API_KEY:
    print("Please add your Groq API key in the GROQ_API_KEY variable")
    print("Get a free key at: https://groq.com")
else:
    groq_client = Groq(api_key=GROQ_API_KEY)
    print("Groq client initialized successfully")

# CBT techniques

In [ ]:
#fetch technique profiles from supabase
def fetch_technique_profiles():
    """fetch all CBT technique profiles from supabase database"""
    try:
        response = supabase.table('cbt_techniques').select('*').execute()
        
        if not response.data:
            raise Exception("no technique profiles found in database")
        
        #convert database rows to TECHNIQUE_PROFILES dictionary
        profiles = {}
        
        for row in response.data:
            #map technique_name to enum
            technique_key = CBTTechnique(row['technique_name'])
            
            profiles[technique_key] = {
                'description': row['description'],
                'example_phrases': row['example_phrases'],
                'indicators': row['indicators'],
                'emotional_states': row['emotional_states'],
                'when_to_use': row['when_to_use'],
                'when_not_to_use': row['when_not_to_use']
            }
        
        return profiles
        
    except Exception as e:
        print(f"error fetching technique profiles: {str(e)}")
        raise

#cbt technique enum
class CBTTechnique(Enum):
    COGNITIVE_RESTRUCTURING = "cognitive_restructuring"
    BEHAVIORAL_ACTIVATION = "behavioral_activation"
    GROUNDING = "grounding"
    PROBLEM_SOLVING = "problem_solving"
    MINDFULNESS = "mindfulness"
    EMOTION_REGULATION = "emotion_regulation"

#fetch profiles from database
TECHNIQUE_PROFILES = fetch_technique_profiles()

#technique distinctions (keeping this hardcoded for now)
TECHNIQUE_DISTINCTIONS = {
    "COGNITIVE_RESTRUCTURING vs MINDFULNESS": 
        "Cognitive restructuring CHALLENGES and CHANGES thoughts; mindfulness OBSERVES and ACCEPTS thoughts without changing them",
    
    "GROUNDING vs MINDFULNESS":
        "Grounding is for ACUTE distress/panic (immediate relief); mindfulness is for CHRONIC rumination/worry (long-term practice)",
    
    "BEHAVIORAL_ACTIVATION vs PROBLEM_SOLVING":
        "Behavioral activation addresses MOOD through activity; problem-solving addresses PRACTICAL issues through planning",
    
    "EMOTION_REGULATION vs GROUNDING":
        "Emotion regulation teaches SKILLS for managing intense emotions; grounding provides IMMEDIATE relief from acute distress"
}

print(f"Loaded {len(TECHNIQUE_PROFILES)} CBT technique profiles from Supabase")

In [ ]:
#initialize sentence transformer for embeddings
print("="*50)
print("Loading embedding model")
print("="*50)

#using a smaller, efficient model that works well for semantic similarity
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

#pre-compute embeddings for all technique example phrases
technique_embeddings = {}

for technique, profile in TECHNIQUE_PROFILES.items():
    #combine description and example phrases for embedding
    texts_to_embed = [profile["description"]] + profile["example_phrases"]
    
    #compute embeddings
    embeddings = embedding_model.encode(texts_to_embed)
    
    #store mean embedding for the technique
    technique_embeddings[technique] = np.mean(embeddings, axis=0)

print(f"Embedding model loaded and {len(technique_embeddings)} technique embeddings computed")

In [ ]:
#embedding-based technique selection
def select_technique_by_embedding(user_input: str, top_k: int = 3) -> List[Tuple[CBTTechnique, float]]:
    """
    Select CBT techniques using semantic similarity
    Returns top_k techniques with confidence scores
    """
    #encode user input
    user_embedding = embedding_model.encode([user_input])[0]
    
    #calculate similarities with all techniques
    similarities = {}
    for technique, technique_embedding in technique_embeddings.items():
        #cosine similarity between user input and technique
        similarity = cosine_similarity(
            user_embedding.reshape(1, -1),
            technique_embedding.reshape(1, -1)
        )[0][0]
        similarities[technique] = similarity
    
    #sort by similarity and get top k
    sorted_techniques = sorted(
        similarities.items(), 
        key=lambda x: x[1], 
        reverse=True
    )[:top_k]
    
    return sorted_techniques

#test embedding selection
test_input = "I feel like everything I do fails"
results = select_technique_by_embedding(test_input)
print(f"Test: '{test_input}'")
for technique, score in results:
    print(f"  {technique.value}: {score:.3f}")

In [ ]:
#safe fallback techniques when all candidates are rejected
SAFE_FALLBACK_TECHNIQUES = [
    CBTTechnique.MINDFULNESS,  #universally safe, broadly applicable, present-moment focus
    CBTTechnique.GROUNDING      #safe for high distress, immediate calming
]

# Checks

In [ ]:
#llm-based scope detection
def detect_scope_with_llm(user_input: str) -> Tuple[str, str, str]:
    """
    use LLM to detect if input is appropriate for CBT chatbot
    returns (scope_category, reason, response_message)
    
    scope_category: "cbt_appropriate", "specialized_mental_health", "not_mental_health"
    """
    scope_prompt = f"""you are a mental health scope detector. your job is to determine if a user's input is appropriate for a CBT chatbot or if it requires a different response.

user input: "{user_input}"

classify this input into ONE of these three categories:

1. CBT_APPROPRIATE: general mental health concerns suitable for CBT chatbot
   - anxiety, worry, stress about situations
   - negative thought patterns, self-criticism
   - depression, low mood, lack of motivation
   - difficulty with emotions or behaviors
   - general psychological distress

2. SPECIALIZED_MENTAL_HEALTH: serious conditions requiring professional care
   - eating disorders (restriction, binging, purging, body image obsession)
   - self-harm or suicidal ideation
   - psychosis, delusions, hallucinations, paranoia
   - mania, extreme mood swings, no sleep for days
   - severe trauma, PTSD flashbacks
   - substance abuse as primary concern
   - severe dissociation or identity issues

3. NOT_MENTAL_HEALTH: casual conversation or unrelated topics
   - greetings ("hi", "hello", "how are you")
   - chitchat or social pleasantries
   - factual questions unrelated to mental health
   - technical questions, general advice
   - sharing positive experiences without distress

CRITICAL: if there's ANY indication of eating disorder behavior, self-harm, or severe symptoms, classify as SPECIALIZED_MENTAL_HEALTH even if the user seems casual about it.

respond in this exact format:
CATEGORY: [CBT_APPROPRIATE/SPECIALIZED_MENTAL_HEALTH/NOT_MENTAL_HEALTH]
REASON: [one sentence explaining why this category fits]
RESPONSE: [if SPECIALIZED_MENTAL_HEALTH, provide crisis referral message. otherwise write "None needed"]

examples:

input: "I'm really anxious about my job interview tomorrow"
CATEGORY: CBT_APPROPRIATE
REASON: user expresses normal anxiety about a specific situation that CBT can help with.
RESPONSE: None needed

input: "I ate too much today so I need to skip meals tomorrow to make up for it"
CATEGORY: SPECIALIZED_MENTAL_HEALTH
REASON: this indicates disordered eating patterns involving restriction and compensation that require specialized eating disorder treatment.
RESPONSE: I'm concerned about what you're sharing. thoughts about compensating for eating through restriction can be part of disordered eating patterns that benefit from specialized support. I'd strongly encourage you to speak with a healthcare provider, therapist specializing in eating disorders, or contact the national eating disorders association helpline at 1-800-931-2237. this is important and you deserve proper support.

input: "what should I eat for dinner tonight?"
CATEGORY: NOT_MENTAL_HEALTH
REASON: this is a casual food choice question with no indication of mental health concerns.
RESPONSE: None needed

input: "I keep hearing voices telling me people are plotting against me"
CATEGORY: SPECIALIZED_MENTAL_HEALTH
REASON: user reports hallucinations and paranoid delusions which require immediate professional psychiatric evaluation.
RESPONSE: what you're experiencing sounds very distressing. hearing voices and feeling that others are plotting against you are symptoms that need professional psychiatric care. please reach out to a mental health professional, call a crisis line, or go to an emergency room if you feel unsafe. you can also call 988 (suicide and crisis lifeline) for immediate support.

input: "hi there! how are you doing today?"
CATEGORY: NOT_MENTAL_HEALTH
REASON: this is a casual greeting with no indication of mental health concerns.
RESPONSE: None needed"""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "user", "content": scope_prompt}
            ],
            temperature=0.2,
            max_tokens=300
        )
        
        response_text = response.choices[0].message.content.strip()
        
        #parse response
        lines = response_text.split('\n')
        category = "cbt_appropriate"
        reason = "classification unclear"
        response_message = ""
        
        for line in lines:
            if line.startswith('CATEGORY:'):
                category_raw = line.replace('CATEGORY:', '').strip().lower()
                if 'specialized' in category_raw:
                    category = "specialized_mental_health"
                elif 'not_mental_health' in category_raw or 'not mental health' in category_raw:
                    category = "not_mental_health"
                else:
                    category = "cbt_appropriate"
            elif line.startswith('REASON:'):
                reason = line.replace('REASON:', '').strip()
            elif line.startswith('RESPONSE:'):
                response_message = line.replace('RESPONSE:', '').strip()
                if response_message.lower() == "none needed":
                    response_message = ""
        
        return category, reason, response_message
        
    except Exception as e:
        return "cbt_appropriate", f"detection error: {str(e)}", ""

print("LLM-based scope detection configured")

In [ ]:
#casual conversation response generator
def generate_casual_response(user_input: str) -> str:
    """
    generate contextual response for non-mental health questions
    provides helpful answer naturally without forced reminders
    """
    casual_prompt = f"""you are a mental health chatbot, but the user has asked you a casual question unrelated to mental health.

user input: "{user_input}"

respond to their question in a brief, natural, and helpful way. just answer their question directly in 1-2 sentences. be warm and conversational.

do NOT remind them you're a mental health chatbot or ask if they have mental health concerns. just answer naturally like a helpful friend would.

examples:

user: "what should I eat for dinner tonight?"
response: that depends on what you're craving! maybe something balanced with protein and veggies if you want to feel satisfied and energized.

user: "what's the weather like?"
response: I don't have access to current weather data, but you can check your weather app for the most accurate forecast in your area.

user: "tell me a joke"
response: why don't scientists trust atoms? because they make up everything! not my best work, but hopefully it got a smile.

user: "how are you?"
response: I'm doing well, thanks for asking! how are you doing today?

now respond to: "{user_input}" """

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "user", "content": casual_prompt}
            ],
            temperature=0.7,
            max_tokens=150
        )
        
        response_text = response.choices[0].message.content.strip()
        
        #remove surrounding quotes if present
        if response_text.startswith('"') and response_text.endswith('"'):
            response_text = response_text[1:-1]
        elif response_text.startswith("'") and response_text.endswith("'"):
            response_text = response_text[1:-1]
        
        return response_text
        
    except Exception as e:
        return "I'm here to chat! what's on your mind?"

print("Casual conversation response generator configured")

In [ ]:
#llm-based ambiguity detection
def detect_ambiguity_with_llm(user_input: str) -> Tuple[bool, str, str]:
    """
    Use LLM to detect if user input is too ambiguous to select a technique
    Returns (is_ambiguous, reason, clarification_response)
    """
    ambiguity_prompt = f"""You are an expert at assessing whether user inputs contain enough information for a mental health chatbot to provide meaningful support.

User Input: "{user_input}"

Question: Is this input clear enough to understand what kind of support the user needs, or is it too vague/ambiguous?

Consider:
- Does the input express a specific thought, feeling, or situation?
- Can you understand what the user is struggling with?
- Is there enough context to provide targeted support?

Inputs that are TOO AMBIGUOUS:
- Single words like "help", "bad", "stressed" without context
- Very vague statements like "I feel weird" or "something's wrong"
- Inputs that could mean many different things

Inputs that are CLEAR ENOUGH:
- Specific emotions or thoughts even if brief ("I'm anxious about my presentation")
- Clear problems even if short ("I can't sleep")
- Specific situations ("my boss yelled at me")

Respond in this exact format:
AMBIGUOUS: [YES/NO]
REASON: [Brief explanation in one sentence]
CLARIFICATION: [If ambiguous, suggest what clarifying question to ask. If not ambiguous, write "None needed"]

Examples:
AMBIGUOUS: YES
REASON: Single word "help" provides no context about what the user needs help with.
CLARIFICATION: I'm here to help. Could you tell me more about what's troubling you right now?

AMBIGUOUS: NO
REASON: User clearly expresses feeling anxious with specific context.
CLARIFICATION: None needed"""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "user", "content": ambiguity_prompt}
            ],
            temperature=0.3,
            max_tokens=200
        )
        
        response_text = response.choices[0].message.content.strip()
        
        #parse response
        lines = response_text.split('\n')
        is_ambiguous = False
        reason = "No reason provided"
        clarification = "Could you tell me more about what's on your mind?"
        
        for line in lines:
            if line.startswith('AMBIGUOUS:'):
                is_ambiguous = 'YES' in line.upper()
            elif line.startswith('REASON:'):
                reason = line.replace('REASON:', '').strip()
            elif line.startswith('CLARIFICATION:'):
                clarification = line.replace('CLARIFICATION:', '').strip()
                if clarification.lower() == "none needed":
                    clarification = ""
        
        return is_ambiguous, reason, clarification
        
    except Exception as e:
        #if detection fails, assume not ambiguous to avoid blocking
        return False, f"Detection error: {str(e)}", ""

print("LLM-based ambiguity detection configured")

# Technique Validation

In [ ]:
def validate_technique_with_llm(user_input: str, technique: CBTTechnique) -> Tuple[bool, str]:
    """
    use LLM to validate if selected technique is appropriate
    returns (is_appropriate, reasoning)
    """
    #get technique profile for enhanced context
    profile = TECHNIQUE_PROFILES[technique]
    
    #get relevant distinctions for this technique
    relevant_distinctions = []
    for comparison, distinction in TECHNIQUE_DISTINCTIONS.items():
        if technique.value.upper() in comparison.upper():
            relevant_distinctions.append(f"- {comparison}: {distinction}")
    
    distinctions_text = "\n".join(relevant_distinctions) if relevant_distinctions else ""
    
    validation_prompt = f"""you are a CBT expert validator. your job is to determine if a selected CBT technique is appropriate for a given user input.

user input: "{user_input}"

selected technique: {technique.value.replace('_', ' ').title()}

technique information:
- description: {profile['description']}
- common indicators: {', '.join(profile['indicators'])}
- typical emotional states: {', '.join(profile['emotional_states'])}
- when to use: {profile['when_to_use']}
- when NOT to use: {profile['when_not_to_use']}

{f'important distinctions:{distinctions_text}' if distinctions_text else ''}

question: is this technique appropriate and safe to use for this user input?

consider:
- is this the most helpful technique for this specific situation?
- are there any safety concerns (e.g., crisis situations)?
- does the technique match the emotional/psychological state implied?
- do any of the "when NOT to use" conditions apply?
- do the indicators and emotional states match the user's situation?
- are there other techniques that would be more appropriate based on the distinctions above?

respond in this exact format:
APPROPRIATE: [YES/NO]
REASONING: [brief explanation in one sentence]

example responses:
APPROPRIATE: YES
REASONING: cognitive restructuring is appropriate for challenging negative self-beliefs.

APPROPRIATE: NO
REASONING: user is experiencing panic symptoms and needs grounding, not cognitive restructuring."""
    
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "user", "content": validation_prompt}
            ],
            temperature=0.3,
            max_tokens=150
        )
        
        response_text = response.choices[0].message.content.strip()
        
        #parse response
        lines = response_text.split('\n')
        is_appropriate = False
        reasoning = "no reasoning provided"
        
        for line in lines:
            if line.startswith('APPROPRIATE:'):
                is_appropriate = 'YES' in line.upper()
            elif line.startswith('REASONING:'):
                reasoning = line.replace('REASONING:', '').strip()
        
        return is_appropriate, reasoning
        
    except Exception as e:
        #if validation fails, be conservative and reject
        return False, f"validation error: {str(e)}"

print("LLM validation function created")

In [ ]:
#enhanced selector with scope detection
class ValidatedTechniqueSelector:
    """Hybrid selector with scope detection, LLM validation, and ambiguity handling"""

    def __init__(self):
        self.technique_history = []
        self.conversation_context = {}
        self.validation_stats = {
            "total_validations": 0,
            "approved": 0,
            "rejected": 0,
            "fallback_used": 0,
            "clarification_requested": 0,
            "out_of_scope": 0
        }

    def select_technique(self, user_input: str) -> Tuple[CBTTechnique, Dict[str, Any], bool, bool]:
        """
        Select technique with scope detection, LLM validation, and ambiguity handling
        Returns (technique, selection_metadata, needs_clarification, out_of_scope)
        """
        #step 1: check scope - is this appropriate for CBT?
        scope_category, scope_reason, scope_response = detect_scope_with_llm(user_input)

        if scope_category != "cbt_appropriate":
            self.validation_stats["out_of_scope"] += 1
            metadata = {
                "scope_category": scope_category,
                "scope_reason": scope_reason,
                "scope_response": scope_response,
                "out_of_scope": True
            }
            return None, metadata, False, True  #out of scope

        #step 2: check for ambiguity using llm
        is_ambiguous, ambiguity_reason, clarification_response = detect_ambiguity_with_llm(user_input)

        if is_ambiguous:
            self.validation_stats["clarification_requested"] += 1
            metadata = {
                "scope_category": scope_category,
                "is_ambiguous": True,
                "ambiguity_reason": ambiguity_reason,
                "clarification_response": clarification_response,
                "out_of_scope": False
            }
            return None, metadata, True, False  #needs clarification

        #step 3: get top 3 techniques from embeddings
        embedding_results = select_technique_by_embedding(user_input, top_k=3)

        #step 4: try to validate each technique in order
        validation_attempts = []
        selected_technique = None

        for technique, similarity_score in embedding_results:
            self.validation_stats["total_validations"] += 1

            #validate with llm
            is_appropriate, reasoning = validate_technique_with_llm(user_input, technique)

            validation_attempts.append({
                "technique": technique,
                "similarity_score": similarity_score,
                "is_appropriate": is_appropriate,
                "reasoning": reasoning
            })

            if is_appropriate:
                self.validation_stats["approved"] += 1
                selected_technique = technique
                break
            else:
                self.validation_stats["rejected"] += 1

        #step 5: if all rejected, validate all fallbacks and choose best
        if selected_technique is None:
            self.validation_stats["fallback_used"] += 1

            #validate all fallbacks
            fallback_validations = []

            for fallback_technique in SAFE_FALLBACK_TECHNIQUES:
                is_appropriate, reasoning = validate_technique_with_llm(user_input, fallback_technique)
                self.validation_stats["total_validations"] += 1

                fallback_validations.append({
                    "technique": fallback_technique,
                    "similarity_score": 0.0,
                    "is_appropriate": is_appropriate,
                    "reasoning": reasoning,
                    "is_fallback": True
                })

                if is_appropriate:
                    self.validation_stats["approved"] += 1
                else:
                    self.validation_stats["rejected"] += 1

            #add fallback validations to attempts
            validation_attempts.extend(fallback_validations)

            #get approved fallbacks
            approved_fallbacks = [fv for fv in fallback_validations if fv["is_appropriate"]]

            if len(approved_fallbacks) == 0:
                #no fallbacks approved, force use first fallback
                selected_technique = SAFE_FALLBACK_TECHNIQUES[0]
                validation_attempts.append({
                    "technique": selected_technique,
                    "similarity_score": 0.0,
                    "is_appropriate": False,
                    "reasoning": "Forced fallback - all options rejected",
                    "is_fallback": True,
                    "is_forced": True
                })
            elif len(approved_fallbacks) == 1:
                #only one approved, use it
                selected_technique = approved_fallbacks[0]["technique"]
            else:
                #multiple fallbacks approved, ask llm to choose best
                selected_technique = self._choose_best_fallback(
                    user_input,
                    approved_fallbacks
                )
                validation_attempts.append({
                    "technique": selected_technique,
                    "similarity_score": 0.0,
                    "is_appropriate": True,
                    "reasoning": "Selected as best among multiple valid fallbacks",
                    "is_fallback": True,
                    "is_best_choice": True
                })

        #update history
        self.technique_history.append(selected_technique)
        if len(self.technique_history) > 10:
            self.technique_history.pop(0)

        #prepare metadata
        metadata = {
            "scope_category": scope_category,
            "embedding_results": embedding_results,
            "validation_attempts": validation_attempts,
            "selected": selected_technique,
            "reasoning": self._generate_reasoning(validation_attempts, selected_technique),
            "validation_stats": self.validation_stats.copy(),
            "is_ambiguous": False,
            "out_of_scope": False
        }

        return selected_technique, metadata, False, False  #no clarification needed, in scope

    def _choose_best_fallback(
        self,
        user_input: str,
        approved_fallbacks: List[Dict]
    ) -> CBTTechnique:
        """Use LLM to choose best technique among multiple approved fallbacks"""
        options = []
        for i, fb in enumerate(approved_fallbacks, 1):
            tech_name = fb["technique"].value.replace('_', ' ').title()
            options.append(f"{i}. {tech_name}: {fb['reasoning']}")

        options_str = "\n".join(options)

        selection_prompt = f"""You are a CBT expert. Multiple techniques are appropriate for this user input, but you must select the MOST appropriate one.

User Input: "{user_input}"

Valid technique options:
{options_str}

Question: Which technique is MOST appropriate for this specific situation?

Respond with just the number (1, 2, etc.) and a brief reason in this format:
SELECTED: [number]
REASON: [one sentence explaining why this is most appropriate]"""

        try:
            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "user", "content": selection_prompt}
                ],
                temperature=0.3,
                max_tokens=100
            )

            response_text = response.choices[0].message.content.strip()

            selected_index = 0
            for line in response_text.split('\n'):
                if line.startswith('SELECTED:'):
                    try:
                        selected_index = int(line.split(':')[1].strip()) - 1
                        break
                    except:
                        selected_index = 0

            if 0 <= selected_index < len(approved_fallbacks):
                return approved_fallbacks[selected_index]["technique"]
            else:
                return approved_fallbacks[0]["technique"]

        except Exception as e:
            return approved_fallbacks[0]["technique"]

    def _generate_reasoning(self, validation_attempts: List, selected: CBTTechnique) -> str:
        """Generate explanation for technique selection"""
        for attempt in validation_attempts:
            if attempt["technique"] == selected:
                if attempt.get("is_forced"):
                    return f"Forced fallback to {selected.value} (all options rejected)"
                elif attempt.get("is_best_choice"):
                    return f"Selected {selected.value} as best among multiple valid fallbacks"
                elif attempt.get("is_fallback"):
                    return f"Fallback to {selected.value}: {attempt['reasoning']}"
                else:
                    return f"Selected {selected.value}: {attempt['reasoning']}"
        return f"Selected {selected.value}"

    def get_validation_stats(self) -> Dict[str, Any]:
        """Get validation statistics"""
        total = self.validation_stats["total_validations"]
        if total == 0:
            return self.validation_stats

        return {
            **self.validation_stats,
            "approval_rate": f"{(self.validation_stats['approved'] / total) * 100:.1f}%" if total > 0 else "0%",
            "rejection_rate": f"{(self.validation_stats['rejected'] / total) * 100:.1f}%" if total > 0 else "0%"
        }

    def reset(self):
        """Reset conversation context"""
        self.technique_history = []
        self.conversation_context = {}

validated_selector = ValidatedTechniqueSelector()
print("Validated technique selector initialized with scope detection")

# Response generation

In [ ]:
#comprehensive cbt system prompt for groq
CBT_SYSTEM_PROMPT = """You are a supportive AI assistant trained in Cognitive Behavioral Therapy (CBT) techniques.

CRITICAL BOUNDARIES - READ FIRST:
You are NOT a licensed therapist, psychologist, or mental health professional. Your role is to:
- Provide emotional support and CBT-based guidance
- Help users explore their thoughts and feelings
- Offer coping strategies and perspectives
- Be a compassionate listener

You CANNOT and DO NOT:
- Diagnose mental health conditions
- Replace professional therapy or medical care
- Provide treatment for serious mental health issues
- Prescribe medications or treatment plans
- Handle crisis situations (these require immediate professional help)

If a user's concerns seem beyond self-help or require professional expertise, gently acknowledge this and encourage them to seek qualified mental health support. You are a complement to professional care, never a replacement.

IMPORTANT: You will be told which specific CBT technique to apply. Follow that instruction.

CORE CBT PRINCIPLES:
1. Collaborative Empiricism: Work WITH the user to examine evidence objectively
2. Socratic Questioning: Guide discovery through thoughtful questions (use this as a METHOD within techniques, not as a standalone response)
3. Present-Focused: Concentrate on current problems and practical solutions
4. Cognitive Restructuring: Help identify and challenge unhelpful thought patterns
5. Behavioral Activation: Encourage engagement in meaningful activities

TECHNIQUE-SPECIFIC GUIDELINES:

For COGNITIVE_RESTRUCTURING:
- Identify the specific negative automatic thought
- Point out cognitive distortions (all-or-nothing, catastrophizing, mind reading, overgeneralization, etc.)
- Examine evidence for and against the thought
- Guide user to question their beliefs through Socratic questions
- Develop a balanced alternative thought
- For rumination, help user see the thought pattern and test its validity

For BEHAVIORAL_ACTIVATION:
- Acknowledge low mood/lack of motivation with empathy
- Suggest small, achievable activities aligned with user's values
- Focus on values-based actions that bring meaning
- Help schedule specific times and create action steps
- Build momentum through gradual engagement

For GROUNDING:
- Guide through 5-4-3-2-1 sensory technique (5 things you see, 4 you touch, etc.)
- Encourage slow, deep breathing exercises
- Focus attention on immediate environment and physical sensations
- Provide calm, steady reassurance
- Use present-tense language to anchor to now

For PROBLEM_SOLVING:
- Help define the problem clearly and specifically
- Generate multiple solution options collaboratively
- Evaluate pros and cons of each option
- Create concrete action steps
- Identify potential obstacles and how to address them

For MINDFULNESS:
- Guide present-moment awareness without judgment
- Teach observation of thoughts and feelings without engaging them
- Encourage acceptance of current experience
- Suggest breathing awareness or body scan practices
- Help distinguish between being IN thoughts vs OBSERVING thoughts
- Address worry and rumination by returning focus to present

For EMOTION_REGULATION:
- Validate the emotion first - all feelings are valid
- Teach healthy coping strategies for intense emotions
- Focus on distress tolerance and emotion acceptance
- Suggest healthy outlets (journaling, physical activity, creative expression)
- Help identify emotion triggers and early warning signs
- Build skills for riding emotional waves without impulsive action

RESPONSE STYLE:
- Be warm, empathetic, and non-judgmental
- Use "we" language for collaboration ("Let's explore this together")
- Keep responses focused, clear, and actionable
- Ask open-ended questions to promote reflection
- Maintain appropriate boundaries - remind users you're not a therapist when relevant
- If concerns seem serious or persistent, gently suggest professional support
- Acknowledge limitations openly and honestly
- Keep responses brief and conversational (2-3 short paragraphs, 80-120 words max)
- Focus on ONE main point or action per response
- Avoid lengthy explanations - users can ask follow-up questions
- End with a single focused question when appropriate

WHEN TO ENCOURAGE PROFESSIONAL HELP:
- User expresses persistent symptoms affecting daily functioning
- Issues seem complex or long-standing
- User asks for diagnosis or medication advice
- Symptoms worsen despite self-help efforts
- User expresses desire for deeper therapeutic work

Example phrases when appropriate:
- "What you're describing sounds like it might benefit from working with a licensed therapist who can provide more comprehensive support."
- "While I can help you explore these thoughts using CBT techniques, a mental health professional could offer more personalized guidance for your situation."
- "I'm here to support you, but I want to be clear that I'm not a replacement for professional therapy, especially for concerns like these."
"""

print("CBT system prompt configured with professional boundaries")

In [ ]:
#generate cbt response
def generate_cbt_response(
    user_input: str, 
    technique: CBTTechnique,
    metadata: Dict[str, Any]
) -> Tuple[str, Dict]:
    """
    Generate CBT response using Groq with selected technique
    """
    #create technique-specific instruction
    technique_instruction = f"""
    Apply the {technique.value.replace('_', ' ').title()} technique.
    {TECHNIQUE_PROFILES[technique]['description']}

    User input: {user_input}
    """
    
    try:
        #call groq api
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            
            messages=[
                {"role": "system", "content": CBT_SYSTEM_PROMPT},
                {"role": "user", "content": technique_instruction}
            ],
            temperature=0.7,
            max_tokens=200
        )
        
        response_text = response.choices[0].message.content
        
        #add technique info to response
        enhanced_response = {
            "response": response_text,
            "technique": technique.value,
            "confidence": metadata["embedding_results"][0][1] if metadata["embedding_results"] else 0,
            "reasoning": metadata["reasoning"]
        }
        
        return response_text, enhanced_response
        
    except Exception as e:
        error_response = f"I apologize, I'm having trouble responding. Error: {str(e)}"
        return error_response, {"error": str(e)}

print("Response generation function configured")

# Chatbot

In [ ]:
#chatbot
class ValidatedCBTChatbot:
    """CBT chatbot with scope detection, LLM-validated technique selection, and ambiguity handling"""
    
    def __init__(self):
        self.selector = ValidatedTechniqueSelector()
        self.conversation_history = []
        self.session_start = datetime.now()
    
    def respond(self, user_input: str) -> Dict[str, Any]:
        """generate complete CBT response with scope detection and validated technique selection"""
        
        #select technique with scope detection, llm validation, and ambiguity check
        technique, metadata, needs_clarification, out_of_scope = self.selector.select_technique(user_input)
        
        #if out of scope, handle based on category
        if out_of_scope:
            scope_category = metadata['scope_category']
            
            #specialized mental health - use fixed crisis referral
            if scope_category == 'specialized_mental_health':
                response_text = metadata["scope_response"]
            
            #not mental health - generate contextual casual response
            elif scope_category == 'not_mental_health':
                response_text = generate_casual_response(user_input)
            
            #fallback
            else:
                response_text = metadata.get("scope_response", "I'm here to help with mental health concerns. is there something specific troubling you?")
            
            result = {
                "user_input": user_input,
                "response": response_text,
                "technique": f"out_of_scope_{scope_category}",
                "timestamp": datetime.now().isoformat(),
                "selection_metadata": metadata,
                "needs_clarification": False,
                "out_of_scope": True
            }
            self.conversation_history.append(result)
            return result
        
        #if needs clarification, return clarification request
        if needs_clarification:
            result = {
                "user_input": user_input,
                "response": metadata["clarification_response"],
                "technique": "clarification",
                "timestamp": datetime.now().isoformat(),
                "selection_metadata": metadata,
                "needs_clarification": True,
                "out_of_scope": False
            }
            self.conversation_history.append(result)
            return result
        
        #generate cbt response using groq
        response_text, response_data = generate_cbt_response(user_input, technique, metadata)
        
        #create complete response object
        result = {
            "user_input": user_input,
            "response": response_text,
            "technique": technique.value,
            "timestamp": datetime.now().isoformat(),
            "selection_metadata": metadata,
            "response_data": response_data,
            "needs_clarification": False,
            "out_of_scope": False
        }
        
        #update history
        self.conversation_history.append(result)
        
        return result
    
    def get_session_summary(self) -> Dict[str, Any]:
        """get summary of conversation session"""
        if not self.conversation_history:
            return {"message": "no conversation yet"}
        
        #analyze techniques used
        techniques_used = [conv["technique"] for conv in self.conversation_history]
        technique_counts = {}
        for tech in techniques_used:
            technique_counts[tech] = technique_counts.get(tech, 0) + 1
        
        #get validation stats
        validation_stats = self.selector.get_validation_stats()
        
        return {
            "session_duration": str(datetime.now() - self.session_start),
            "total_interactions": len(self.conversation_history),
            "techniques_used": technique_counts,
            "most_common_technique": max(technique_counts, key=technique_counts.get) if technique_counts else None,
            "validation_statistics": validation_stats
        }
    
    def reset(self):
        """reset conversation"""
        self.selector.reset()
        self.conversation_history = []
        self.session_start = datetime.now()
        print("conversation reset")

validated_chatbot = ValidatedCBTChatbot()
print("Validated CBT chatbot initialized with dynamic casual responses")

# Test

In [ ]:
#test the validated system
def test_validated_system():
    """Test the LLM-validated CBT system"""
    
    test_cases = [
        "I'm a complete failure at everything",
        "I can't breathe, my heart is racing, I think I'm having a panic attack",
        "Nothing brings me joy anymore, I just stay in bed all day",
        "I don't know how to handle this situation at work",
        "Why do I always feel so anxious about everything?"
    ]
    
    print("="*70)
    print("TESTING LLM-VALIDATED CBT TECHNIQUE SELECTION")
    print("="*70)
    
    for i, test_input in enumerate(test_cases, 1):
        print(f"\n{'='*70}")
        print(f"[Test {i}] User: {test_input}")
        print('='*70)
        
        #get response
        result = validated_chatbot.respond(test_input)

        #check if out of scope
        if result.get('out_of_scope'):
            print("\nOUT OF SCOPE DETECTED:")
            print(f"Category: {result['selection_metadata']['scope_category']}")
            print(f"Reason: {result['selection_metadata']['scope_reason']}")
            print(f"\nResponse:")
            print(f"{result['response']}")
            print()
            continue
        
        #check if clarification was needed
        if result.get('needs_clarification'):
            print("\nAMBIGUITY DETECTED:")
            print(f"Reason: {result['selection_metadata']['ambiguity_reason']}")
            print(f"\nClarification Response:")
            print(f"{result['response']}")
            print()
            continue
        
        #display selection process
        print("\nTECHNIQUE SELECTION PROCESS:")
        print(f"Top 3 from embeddings:")
        for j, (tech, score) in enumerate(result['selection_metadata']['embedding_results'], 1):
            print(f"  {j}. {tech.value}: similarity={score:.3f}")
        
        print(f"\nValidation attempts:")
        for j, attempt in enumerate(result['selection_metadata']['validation_attempts'], 1):
            tech = attempt['technique'].value
            appropriate = "APPROVED" if attempt['is_appropriate'] else "REJECTED"
            fallback = " [FALLBACK]" if attempt.get('is_fallback') else ""
            forced = " [FORCED]" if attempt.get('is_forced') else ""
            print(f"  {j}. {tech}: {appropriate}{fallback}{forced}")
            print(f"     Reasoning: {attempt['reasoning']}")
        
        #display final selection
        print(f"\n>>> FINAL SELECTION: {result['technique']}")
        print(f">>> REASONING: {result['selection_metadata']['reasoning']}")
        
        #display response
        print(f"\nCBT RESPONSE:")
        print(f"{result['response']}")
        
        print()
    
    #show session summary
    print("\n" + "="*70)
    print("SESSION SUMMARY")
    print("="*70)
    summary = validated_chatbot.get_session_summary()
    for key, value in summary.items():
        print(f"{key}: {value}")

#run tests
test_validated_system()

In [ ]:
#interactive function for using the chatbot
def chat_with_cbt():
    """Interactive CBT chat session"""
    
    print("\n" + "="*60)
    print("CBT CHATBOT - Validated Technique Selection")
    print("="*60)
    print("Type 'quit' to exit | 'summary' for stats | 'reset' to start over")
    print("-"*60)
    
    while True:
        #get user input
        user_input = input("\nYou: ").strip()
        
        #handle commands
        if user_input.lower() == 'quit':
            print("\nTake care! Remember, professional support is always available.")
            break
        elif user_input.lower() == 'summary':
            summary = validated_chatbot.get_session_summary()
            print("\nSession Summary:")
            for key, value in summary.items():
                print(f"  {key}: {value}")
            continue
        elif user_input.lower() == 'reset':
            validated_chatbot.reset()
            continue
        
        #get response
        result = validated_chatbot.respond(user_input)
        
        #display response
        print(f"\nCBT Bot [{result['technique']}]:")
        print(result['response'])
        
        #show appropriate metadata based on response type
        if result.get('out_of_scope'):
            print(f"\nScope Detection: {result['selection_metadata']['scope_reason']}")
        elif result.get('needs_clarification'):
            print(f"\nAmbiguity: {result['selection_metadata']['ambiguity_reason']}")
        else:
            #only show technique selection reasoning for actual cbt responses
            print(f"\nTechnique Selection: {result['selection_metadata']['reasoning']}")

print("\nChatbot ready! Call chat_with_cbt() to start interactive session")

In [ ]:
chat_with_cbt()